In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [2]:
# Generate data
# n_points = 300_000
n_points = 1_000
X = np.random.RandomState(42).normal(size=(n_points, 256))
X = X / np.linalg.norm(X, axis=1, keepdims=True)

X = pd.DataFrame(data=X)
X.index = X.index.map(lambda i: f"point_{i}")
X.columns = X.columns.map(lambda j: f"z_{j}")

X_train, X_test = train_test_split(X, test_size=0.15, random_state=42)
X_train.shape, X_test.shape

((850, 256), (150, 256))

In [3]:

# # Create boolean masks based on membership in test set
# train_mask = np.isin(X.index, X_train.index)
# test_mask = np.isin(X.index, X_test.index)

# # Plot 1: Full data colored by true class
# fig, ax = plt.subplots(figsize=(6, 5))
# X.plot(kind="scatter", x="x", y="y", colormap="Set2", ax=ax)
# ax.set_title("Full Data by Class")
# plt.show()

# # Plot 2: Train/test overlay (train=blue, test=red)
# fig, ax = plt.subplots(figsize=(6, 5))
# X.plot(kind="scatter", x="x", y="y", c=test_mask.astype(int), 
#        cmap="RdYlBu", ax=ax)  # 0=train (blue), 1=test (red)
# ax.set_title("Train vs Test Split")
# plt.show()

# # Plot 3: Classes separately for train/test
# fig, axes = plt.subplots(1, 3, figsize=(15, 5))
# for i, class_label in enumerate([0, 1, 2]):
#     mask = y == class_label
#     axes[i].scatter(X.loc[mask & train_mask, "x"], X.loc[mask & train_mask, "y"], 
#                     c="blue", label="train", alpha=0.6)
#     axes[i].scatter(X.loc[mask & test_mask, "x"], X.loc[mask & test_mask, "y"], 
#                     c="red", label="test", alpha=0.6)
#     axes[i].set_title(f"Class {class_label}")
#     axes[i].legend()
# plt.tight_layout()
# plt.show()


In [13]:
import faiss
n = X_train.shape[0]
d = X_train.shape[1]

k = 100
index_flat = faiss.IndexFlatIP(d)
index_flat.add(X_train.values.astype("float32"))
D, I = index_flat.search(X_train.values.astype("float32"), k)


In [4]:
import igraph as ig

index = X_train.index

edges = list()
for i, distances in enumerate(D):
    source_node = index[i]
    neighbors_index = I[i]
    neighbor_nodes = index[neighbors_index]
    for neighbor_node, distance in zip(neighbor_nodes, distances):
        edges.append((source_node, neighbor_node, distance))
graph = ig.Graph.TupleList(edges, weights=True)
graph.summary()


'IGRAPH UNW- 850 85000 -- \n+ attr: name (v), weight (e)'

In [7]:
graph.es["weight"]

[1.0,
 0.18696178,
 0.18460666,
 0.18085773,
 0.16588277,
 0.16538256,
 0.1601076,
 0.15643328,
 0.15465878,
 0.15132613,
 0.14558236,
 0.14446515,
 0.14350984,
 0.14026514,
 0.13711748,
 0.13445996,
 0.13163431,
 0.13146368,
 0.13071918,
 0.13062239,
 0.12926656,
 0.12908994,
 0.1269954,
 0.12414581,
 0.12353559,
 0.120794594,
 0.120762855,
 0.120089516,
 0.11982057,
 0.1180886,
 0.117961444,
 0.1152373,
 0.1148101,
 0.11369733,
 0.1127552,
 0.111248024,
 0.10964054,
 0.10958695,
 0.109319955,
 0.109239645,
 0.106800474,
 0.10613949,
 0.10610006,
 0.10563936,
 0.10354362,
 0.103181735,
 0.10257976,
 0.10252222,
 0.10211165,
 0.101715036,
 0.10133713,
 0.10080316,
 0.10074847,
 0.10018416,
 0.100087196,
 0.0995298,
 0.099409044,
 0.09925024,
 0.098257184,
 0.098032594,
 0.09677547,
 0.09580656,
 0.095805824,
 0.0956363,
 0.09513926,
 0.09507338,
 0.09504062,
 0.0946889,
 0.09468625,
 0.09449809,
 0.09406535,
 0.093682244,
 0.09322943,
 0.09200599,
 0.09128388,
 0.089776225,
 0.08942743

In [30]:
import numpy as np
import warnings
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted, check_array


class KNeighborsCosineSimilarity(BaseEstimator, TransformerMixin):
    """
    K-Nearest Neighbors using cosine similarity.
    
    Parameters
    ----------
    n_neighbors : int
        Number of neighbors to find
    mode : {'exact', 'ivf', 'pq'}, default='exact'
        Search strategy:
        - 'exact': Brute force exact search
        - 'ivf': Inverted file index (approximate)
        - 'pq': Product quantization (compressed, approximate)
    backend : {'auto', 'faiss', 'sklearn'}, default='auto'
        Which library to use. 'auto' tries FAISS, falls back to sklearn
    n_voronoi_cells : int or 'auto', default='auto'
        Number of IVF cells. If 'auto', uses sqrt(n_samples)
    n_probes : int, default=1
        Number of cells to search in IVF (FAISS default is 1)
    n_subvectors : int or None, default=None
        Number of sub-vectors for PQ. If None, uses d//16
    n_bits : int, default=8
        Bits per sub-vector for PQ
    
    Attributes
    ----------
    backend_ : str
        Actual backend used ('faiss' or 'sklearn')
    index_ : faiss index or None
        Fitted FAISS index (None if using sklearn)
    similarities_ : np.ndarray, shape (n_samples_fit, n_neighbors)
        Cosine similarities to k nearest neighbors (higher = more similar)
    indices_ : np.ndarray, shape (n_samples_fit, n_neighbors)
        Indices of k nearest neighbors
    """
    
    def __init__(
        self,
        n_neighbors,
        mode='exact',
        backend='auto',
        n_voronoi_cells='auto',
        n_probes=1,
        n_subvectors=None,
        n_bits=8,
    ):
        self.n_neighbors = n_neighbors
        self.mode = mode
        self.backend = backend
        self.n_voronoi_cells = n_voronoi_cells
        self.n_probes = n_probes
        self.n_subvectors = n_subvectors
        self.n_bits = n_bits
    
    def _determine_backend(self):
        """Determine which backend to use."""
        if self.backend == 'sklearn':
            return 'sklearn'
        elif self.backend == 'faiss':
            try:
                import faiss
                return 'faiss'
            except ImportError:
                raise ImportError("FAISS not available")
        else:  # auto
            try:
                import faiss
                return 'faiss'
            except ImportError:
                warnings.warn("FAISS not available, falling back to sklearn", UserWarning)
                return 'sklearn'
    
    def fit(self, X, y=None):
        """
        Fit the k-NN model.
        
        Parameters
        ----------
        X : array-like, shape (n_samples, n_features)
            Training data (must be L2-normalized for cosine similarity)
        y : Ignored
        
        Returns
        -------
        self : object
        """
        X = check_array(X, dtype=np.float32, ensure_2d=True)
        
        self.n_samples_fit_ = X.shape[0]
        self.n_features_in_ = X.shape[1]
        self.backend_ = self._determine_backend()
        
        if self.backend_ == 'faiss':
            self._fit_faiss(X)
        else:
            self._fit_sklearn(X)
        
        # Store the training data neighbors
        self.similarities_, self.indices_ = self.transform(X)
        
        return self
    
    def _fit_faiss(self, X):
        """Fit using FAISS backend."""
        import faiss
        
        d = self.n_features_in_
        
        if self.mode == 'exact':
            self.index_ = faiss.IndexFlatIP(d)
            self.index_.add(X)
        
        elif self.mode == 'ivf':
            # Determine n_voronoi_cells
            if self.n_voronoi_cells == 'auto':
                nlist = int(np.sqrt(self.n_samples_fit_))
            else:
                nlist = self.n_voronoi_cells
            
            quantizer = faiss.IndexFlatIP(d)
            self.index_ = faiss.IndexIVFFlat(quantizer, d, nlist)
            self.index_.train(X)
            self.index_.add(X)
            self.index_.nprobe = self.n_probes
        
        elif self.mode == 'pq':
            # Determine n_subvectors
            if self.n_subvectors is None:
                m = d // 16
                while d % m != 0 and m > 1:
                    m -= 1
                if m == 1:
                    raise ValueError(
                        f"Cannot determine n_subvectors for dimension {d}. "
                        f"Please specify n_subvectors that divides {d} evenly."
                    )
            else:
                m = self.n_subvectors
                if d % m != 0:
                    raise ValueError(f"n_subvectors ({m}) must divide dimension ({d}) evenly")
            
            self.index_ = faiss.IndexPQ(d, m, self.n_bits)
            self.index_.train(X)
            self.index_.add(X)
        
        else:
            raise ValueError(f"mode must be 'exact', 'ivf', or 'pq', got '{self.mode}'")
    
    def _fit_sklearn(self, X):
        """Fit using sklearn backend."""
        if self.mode != 'exact':
            warnings.warn(
                f"sklearn backend only supports exact search, ignoring mode='{self.mode}'",
                UserWarning
            )
        self.X_fit_ = X
        self.index_ = None
    
    def transform(self, X):
        """
        Find k-nearest neighbors.
        
        Parameters
        ----------
        X : array-like, shape (n_samples, n_features)
            Query vectors (must be L2-normalized)
        
        Returns
        -------
        similarities : np.ndarray, shape (n_samples, n_neighbors)
            Cosine similarities to k nearest neighbors (higher = more similar)
        indices : np.ndarray, shape (n_samples, n_neighbors)
            Indices of k nearest neighbors
        """
        check_is_fitted(self)
        X = check_array(X, dtype=np.float32, ensure_2d=True)
        
        if X.shape[1] != self.n_features_in_:
            raise ValueError(f"X has {X.shape[1]} features, expected {self.n_features_in_}")
        
        if self.n_neighbors > self.n_samples_fit_:
            raise ValueError(f"n_neighbors ({self.n_neighbors}) > n_samples ({self.n_samples_fit_})")
        
        if self.backend_ == 'faiss':
            similarities, indices = self.index_.search(X, self.n_neighbors)
        else:
            from sklearn.neighbors import NearestNeighbors
            nn = NearestNeighbors(n_neighbors=self.n_neighbors, metric='cosine')
            nn.fit(self.X_fit_)
            distances, indices = nn.kneighbors(X)
            similarities = 1 - distances  # Convert distance to similarity
        
        return similarities, indices
    
    def fit_transform(self, X, y=None):
        """Fit and transform in one step."""
        return self.fit(X, y).transform(X)
    
    def to_igraph(self, index=None, include_self=False):
        """
        Convert fitted k-NN results to igraph.
        
        Parameters
        ----------
        index : array-like or None, default=None
            Node labels. If None, uses integers 0 to n-1
        include_self : bool, default=False
            Whether to include self-loops
        
        Returns
        -------
        ig.Graph
            Directed graph with edges weighted by cosine similarity
        """
        check_is_fitted(self, ['similarities_', 'indices_'])
        
        return kneighbors_to_igraph(
            self.similarities_,
            self.indices_,
            index=index,
            include_self=include_self
        )
        """Fit and transform in one step."""
        return self.fit(X, y).transform(X)

In [26]:
def kneighbors_to_igraph(D, I, index=None, include_self=False):
    """
    Convert k-nearest neighbors results to igraph.
    
    Parameters
    ----------
    D : np.ndarray, shape (n, k)
        Cosine similarities to k nearest neighbors (higher = more similar)
    I : np.ndarray, shape (n, k)
        Indices of k nearest neighbors
    index : array-like or None, default=None
        Node labels/IDs. If None, uses integer indices 0, 1, ..., n-1
    include_self : bool, default=False
        Whether to include self-loops
    
    Returns
    -------
    ig.Graph
        Directed graph with edges weighted by cosine similarity
    """
    import igraph as ig
    n, k = I.shape
    
    if not include_self:
        I = I[:, 1:]
        D = D[:, 1:]
        k = k - 1
    
    # Vectorized edge construction
    sources = np.repeat(np.arange(n), k)
    targets = I.flatten()
    weights = D.flatten()
    
    # Map to node labels only if index is provided and non-scalar
    if index is not None:
        index_array = np.asarray(index)
        if index_array.ndim > 0:  # Check it's actually an array
            sources = index_array[sources]
            targets = index_array[targets]
    
    # Create edge list
    edges = list(zip(sources, targets, weights))
    
    graph = ig.Graph.TupleList(edges, weights=True)
    return graph

graph2 = kneighbors_to_igraph(D, I, index)
graph2.es["weight"]

[0.18696178,
 0.18460666,
 0.18085773,
 0.16588277,
 0.16538256,
 0.1601076,
 0.15643328,
 0.15465878,
 0.15132613,
 0.14558236,
 0.14446515,
 0.14350984,
 0.14026514,
 0.13711748,
 0.13445996,
 0.13163431,
 0.13146368,
 0.13071918,
 0.13062239,
 0.12926656,
 0.12908994,
 0.1269954,
 0.12414581,
 0.12353559,
 0.120794594,
 0.120762855,
 0.120089516,
 0.11982057,
 0.1180886,
 0.117961444,
 0.1152373,
 0.1148101,
 0.11369733,
 0.1127552,
 0.111248024,
 0.10964054,
 0.10958695,
 0.109319955,
 0.109239645,
 0.106800474,
 0.10613949,
 0.10610006,
 0.10563936,
 0.10354362,
 0.103181735,
 0.10257976,
 0.10252222,
 0.10211165,
 0.101715036,
 0.10133713,
 0.10080316,
 0.10074847,
 0.10018416,
 0.100087196,
 0.0995298,
 0.099409044,
 0.09925024,
 0.098257184,
 0.098032594,
 0.09677547,
 0.09580656,
 0.095805824,
 0.0956363,
 0.09513926,
 0.09507338,
 0.09504062,
 0.0946889,
 0.09468625,
 0.09449809,
 0.09406535,
 0.093682244,
 0.09322943,
 0.09200599,
 0.09128388,
 0.089776225,
 0.089427434,
 0.

In [32]:
import numpy as np
from sklearn.datasets import make_blobs

print("="*80)
print("Testing KNeighborsCosineSimilarity")
print("="*80)

# Generate and normalize data
print("\n1. Generating test data...")
X, y = make_blobs(n_samples=10000, centers=3, n_features=128, random_state=42)
X = X / np.linalg.norm(X, axis=1, keepdims=True)
print(f"   Data shape: {X.shape}")

# Test 1: Exact search with FAISS
print("\n2. Testing exact search (FAISS backend)...")
knn_exact = KNeighborsCosineSimilarity(n_neighbors=10, backend='faiss')
knn_exact.fit(X)
print(f"   Backend: {knn_exact.backend_}")
print(f"   Similarities shape: {knn_exact.similarities_.shape}")
print(f"   Indices shape: {knn_exact.indices_.shape}")
print(f"   First point's top 3 neighbors: {knn_exact.indices_[0, :3]}")
print(f"   First point's top 3 similarities: {knn_exact.similarities_[0, :3]}")

# Test 2: sklearn backend
print("\n3. Testing exact search (sklearn backend)...")
knn_sklearn = KNeighborsCosineSimilarity(n_neighbors=10, backend='sklearn')
knn_sklearn.fit(X)
print(f"   Backend: {knn_sklearn.backend_}")
print(f"   Results match FAISS: {np.allclose(knn_sklearn.similarities_, knn_exact.similarities_, atol=1e-5)}")

# Test 3: IVF mode
print("\n4. Testing IVF mode...")
knn_ivf = KNeighborsCosineSimilarity(n_neighbors=10, mode='ivf')
knn_ivf.fit(X)
print(f"   Backend: {knn_ivf.backend_}")
print(f"   Mode: {knn_ivf.mode}")
print(f"   Similarities shape: {knn_ivf.similarities_.shape}")

# Test 4: PQ mode
print("\n5. Testing PQ mode...")
knn_pq = KNeighborsCosineSimilarity(n_neighbors=10, mode='pq')
knn_pq.fit(X)
print(f"   Backend: {knn_pq.backend_}")
print(f"   Mode: {knn_pq.mode}")
print(f"   Similarities shape: {knn_pq.similarities_.shape}")

# Test 5: Transform on new data
print("\n6. Testing transform on new data...")
X_new = np.random.randn(5, 128).astype('float32')
X_new = X_new / np.linalg.norm(X_new, axis=1, keepdims=True)
similarities_new, indices_new = knn_exact.transform(X_new)
print(f"   New similarities shape: {similarities_new.shape}")
print(f"   New indices shape: {indices_new.shape}")

# Test 6: fit_transform
print("\n7. Testing fit_transform...")
knn_ft = KNeighborsCosineSimilarity(n_neighbors=10)
similarities_ft, indices_ft = knn_ft.fit_transform(X)
print(f"   fit_transform matches fit().transform(): {np.allclose(similarities_ft, knn_exact.similarities_)}")

# Test 7: to_igraph
print("\n8. Testing to_igraph...")
try:
    import igraph as ig
    graph = knn_exact.to_igraph()
    print(f"   Graph created: {graph.vcount()} vertices, {graph.ecount()} edges")
    
    # Test with custom index
    node_ids = [f"node_{i}" for i in range(len(X))]
    graph_named = knn_exact.to_igraph(index=node_ids)
    print(f"   Graph with custom names: {graph_named.vs[0]['name']}")
    
    # Test without self-loops
    graph_no_self = knn_exact.to_igraph(include_self=False)
    print(f"   Graph without self-loops: {graph_no_self.ecount()} edges")
except ImportError:
    print("   igraph not installed, skipping graph tests")

# Test 8: Custom IVF parameters
print("\n9. Testing custom IVF parameters...")
knn_custom = KNeighborsCosineSimilarity(
    n_neighbors=10,
    mode='ivf',
    n_voronoi_cells=20,
    n_probes=5
)
knn_custom.fit(X)
print(f"   Custom IVF fitted successfully")

print("\n" + "="*80)
print("All tests passed!")
print("="*80)

Testing KNeighborsCosineSimilarity

1. Generating test data...
   Data shape: (10000, 128)

2. Testing exact search (FAISS backend)...
   Backend: faiss
   Similarities shape: (10000, 10)
   Indices shape: (10000, 10)
   First point's top 3 neighbors: [   0 9278 2528]
   First point's top 3 similarities: [0.9999999  0.98209065 0.98134977]

3. Testing exact search (sklearn backend)...
   Backend: sklearn
   Results match FAISS: True

4. Testing IVF mode...
   Backend: faiss
   Mode: ivf
   Similarities shape: (10000, 10)

5. Testing PQ mode...
   Backend: faiss
   Mode: pq
   Similarities shape: (10000, 10)

6. Testing transform on new data...
   New similarities shape: (5, 10)
   New indices shape: (5, 10)

7. Testing fit_transform...
   fit_transform matches fit().transform(): True

8. Testing to_igraph...
   Graph created: 10000 vertices, 90000 edges
   Graph with custom names: node_0
   Graph without self-loops: 90000 edges

9. Testing custom IVF parameters...
   Custom IVF fitted s

First neighbor match rate: 1.0000

Query 0 - Top 5 neighbors:
FAISS:   [   0 8344 3348 7609 6015]
sklearn: [   0 8344 3348 7609 6015]

IVF auto parameters:
n_voronoi_cells: 100 = 100
n_probes: 10 = 10

Query 0 IVF recall@10: 2/10 neighbors found
True:  [0, 4905, 3577, 9874, 3348]...
IVF:   [0, 1891, 5028, 7112, 2250]...
Query 0 manual calculation:
True neighbors: [0, 3348, 3577, 4905, 5269, 6015, 7609, 8344, 8923, 9874]
IVF neighbors:  [0, 1891, 2250, 3577, 3861, 5028, 7112, 7924, 8437, 9371]
Overlap: 2/10 = 0.2000

Mean recall across all queries: 0.1925
